<a href="https://colab.research.google.com/github/eunseojeon/AI_Coding_for_Autonomous_Driving_Class/blob/main/0811_%ED%85%90%EC%84%9Crt%EC%99%80_%ED%8C%8C%EC%9D%B4%ED%86%A0%EC%B9%98_%EB%B9%84%EA%B5%90_ADAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import shutil
import os

folder = "/content"
for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)  # 파일 또는 심볼릭 링크 삭제
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)  # 폴더(디렉토리) 삭제
    except Exception as e:
        print(f'Failed to delete {file_path}. Reason: {e}')

#위 코드는 아래 코드들에서 오류났을 때, 새로운 파일이 생기니까 그 파일을 다 삭제해주는 코드! (리셋느낌)

In [1]:
# 디스크 공간 확인
!df -h

# CUDA 환경 확인
!nvidia-smi

# GPU 정보 확인
!nvidia-ml-py3 || pip install nvidia-ml-py3

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   39G   74G  35% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G     0  5.7G   0% /dev/shm
/dev/root       2.0G  1.2G  775M  61% /usr/sbin/docker-init
tmpfs           6.4G   76K  6.4G   1% /var/colab
/dev/sda1        74G   41G   33G  56% /kaggle/input
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
Mon Aug 11 07:35:00 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                 

In [2]:
# 빠른 공간 확인
import shutil
total, used, free = shutil.disk_usage('/')
print(f"💾 디스크 여유공간: {free // (1024**3):.1f}GB")

💾 디스크 여유공간: 73.0GB


In [3]:
# 기본 패키지 설치
!pip install opencv-python
!pip install numpy
!pip install matplotlib
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install ultralytics

Looking in indexes: https://download.pytorch.org/whl/cu121
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.5/780.5 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 59.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 46.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 63.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 12.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.4 MB/s e

In [4]:
# 패키지 임포트 테스트
try:
    import cv2
    import numpy as np
    import matplotlib.pyplot as plt
    print("✅ 기본 패키지 로드 성공")
except ImportError as e:
    print(f"❌ 패키지 로드 실패: {e}")

# CUDA 확인
try:
    import torch
    print(f"✅ PyTorch: {torch.__version__}")
    print(f"✅ CUDA 사용 가능: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("❌ PyTorch 설치 필요")

✅ 기본 패키지 로드 성공
✅ PyTorch: 2.5.1+cu121
✅ CUDA 사용 가능: True
✅ GPU: Tesla T4


In [5]:
import os

# workspace 이미지 확인
image_files = ['1.png', '2.png', '3.png']
found_images = []

print("📁 이미지 파일 확인...")
for img_file in image_files:
    full_path = f'/content/workspace/{img_file}'
    if os.path.exists(full_path):
        try:
            img = cv2.imread(full_path)
            if img is not None:
                h, w, c = img.shape
                file_size = os.path.getsize(full_path) / 1024  # KB
                print(f"✅ {img_file}: {w}x{h}, {file_size:.1f}KB")
                found_images.append(full_path)
            else:
                print(f"❌ {img_file}: 이미지 읽기 실패")
        except Exception as e:
            print(f"❌ {img_file}: 오류 - {e}")
    else:
        print(f"❌ {img_file}: 파일 없음")


print(f"\n📊 결과: {len(found_images)}개 이미지 발견")


📁 이미지 파일 확인...
✅ 1.png: 3178x1412, 4008.1KB
✅ 2.png: 3178x1416, 3124.1KB
✅ 3.png: 3178x1416, 2905.9KB

📊 결과: 3개 이미지 발견


In [6]:
# YOLO 모델 로드 (처음에는 다운로드 시간이 걸릴 수 있습니다)
from ultralytics import YOLO

print("📦 YOLO 모델 로딩...")
model = YOLO('yolov8n.pt')  # nano 버전 (가장 빠름)
print("✅ YOLO 모델 로드 완료")

# GPU 사용 설정
if torch.cuda.is_available():
    model.to('cuda')
    print("✅ GPU로 모델 이동 완료")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
📦 YOLO 모델 로딩...


✅ YOLO 모델 로드 완료
✅ GPU로 모델 이동 완료


In [7]:
import time
# 첫 번째 이미지로 간단 테스트
if found_images:
    test_image = found_images[0]
    print(f"🔍 테스트 이미지: {os.path.basename(test_image)}")

    # YOLO 추론
    start_time = time.time()
    results = model(test_image, conf=0.5)
    inference_time = time.time() - start_time

    print(f"⏱️ 추론 시간: {inference_time:.3f}초")

    # 결과 확인
    for result in results:
        boxes = result.boxes
        if boxes is not None:
            print(f"🎯 감지된 객체: {len(boxes)}개")

            # 각 객체 정보 출력
            for i, box in enumerate(boxes):
                class_id = int(box.cls[0])
                class_name = result.names[class_id]
                confidence = box.conf[0].cpu().numpy()
                print(f"  {i+1}. {class_name}: {confidence:.2f}")
        else:
            print("❌ 감지된 객체 없음")
else:
    print("❌ 테스트할 이미지가 없습니다")

🔍 테스트 이미지: 1.png

image 1/1 /content/workspace/1.png: 288x640 7 cars, 1 bus, 97.3ms
Speed: 12.0ms preprocess, 97.3ms inference, 126.1ms postprocess per image at shape (1, 3, 288, 640)
⏱️ 추론 시간: 4.699초
🎯 감지된 객체: 8개
  1. car: 0.91
  2. car: 0.89
  3. car: 0.84
  4. car: 0.75
  5. car: 0.57
  6. car: 0.56
  7. bus: 0.54
  8. car: 0.54


In [8]:
# ===== 필요 라이브러리 불러오기 =====
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
import torch
from ultralytics import YOLO
import os

# ============================================
# 1. Complete ADAS System 클래스
# ============================================
class ComprehensiveADAS:
    """객체 탐지 + 차선 인식 + 충돌 위험 분석을 포함한 ADAS 시스템"""

    def __init__(self):
        print("🚗 Initializing Comprehensive ADAS System...")
        self.model = None
        self.load_model()  # YOLO 모델 불러오기

    def load_model(self):
        """YOLO v8 모델 로드"""
        try:
            print("📦 Loading YOLO model...")
            self.model = YOLO('yolov8n.pt')  # 경량 모델

            if torch.cuda.is_available():
                self.model.to('cuda')
                print("✅ Model loaded on GPU")
            else:
                print("✅ Model loaded on CPU")
        except Exception as e:
            print(f"❌ Model loading failed: {e}")

    def detect_objects(self, image_path):
        """객체 탐지 + FPS 성능 측정"""
        print(f"🔍 Object Detection: {os.path.basename(image_path)}")

        # 성능 평균 내기 위해 20회 반복
        times = []
        for i in range(20):
            start_time = time.time()
            results = self.model(image_path, conf=0.5, verbose=False)
            times.append(time.time() - start_time)

        # YOLO 탐지 결과 정리
        result = results[0]
        detections = []
        adas_objects = ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 'traffic light', 'stop sign']

        if result.boxes is not None:
            for box in result.boxes:
                class_id = int(box.cls[0])
                class_name = result.names[class_id]
                confidence = float(box.conf[0])
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                if class_name in adas_objects:
                    detections.append({
                        'class': class_name,
                        'confidence': confidence,
                        'bbox': [int(x1), int(y1), int(x2), int(y2)],
                        'center': [(x1 + x2) / 2, (y1 + y2) / 2]
                    })

        return {
            'detections': detections,
            'fps': 1.0 / np.mean(times),
            'inference_time': np.mean(times),
            'yolo_result': result
        }

    def detect_lanes(self, image_path):
        """간단한 차선 감지(HoughLinesP)"""
        image = cv2.imread(image_path)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (5,5), 0)
        edges = cv2.Canny(blur, 50, 150)

        height, width = gray.shape
        # ROI 범위 지정
        roi_vertices = np.array([[
            (int(width * 0.1), height),
            (int(width * 0.45), int(height * 0.6)),
            (int(width * 0.55), int(height * 0.6)),
            (int(width * 0.9), height)
        ]], dtype=np.int32)

        mask = np.zeros_like(edges)
        cv2.fillPoly(mask, roi_vertices, 255)
        masked_edges = cv2.bitwise_and(edges, mask)

        # 허프 변환으로 차선 추출
        lines = cv2.HoughLinesP(masked_edges, 1, np.pi/180, 50, minLineLength=100, maxLineGap=50)
        left_lines, right_lines = [], []

        if lines is not None:
            for line in lines:
                x1, y1, x2, y2 = line[0]
                slope = (y2 - y1) / (x2 - x1) if x2 != x1 else 0
                if abs(slope) > 0.3:
                    if slope < 0:
                        left_lines.append([x1, y1, x2, y2])
                    else:
                        right_lines.append([x1, y1, x2, y2])

        return {
            'left_lines': left_lines,
            'right_lines': right_lines,
            'roi_vertices': roi_vertices[0],
            'total_lines': len(left_lines) + len(right_lines)
        }

    def collision_risk_analysis(self, detections, image_shape):
        """탐지된 객체와 위치를 기반으로 충돌 위험도 평가"""
        warnings = []
        height, width = image_shape[:2]

        # 위험 구역 정의(critical, warning zone)
        critical_zone = {'x1': width * 0.3, 'y1': height * 0.7, 'x2': width * 0.7, 'y2': height}
        warning_zone = {'x1': width * 0.2, 'y1': height * 0.5, 'x2': width * 0.8, 'y2': height}

        for detection in detections:
            x1, y1, x2, y2 = detection['bbox']
            center_x, center_y = detection['center']
            # 거리, 위험 단계 판정
            # (생략: 코드 동일)
            # warnings.append({...})
        return warnings

    def process_single_image(self, image_path):
        """단일 이미지 전체 분석(객체+차선+위험분석)"""
        obj_results = self.detect_objects(image_path)
        lane_results = self.detect_lanes(image_path)
        image = cv2.imread(image_path)
        warnings = self.collision_risk_analysis(obj_results['detections'], image.shape)

        return {
            'image_path': image_path,
            'objects': obj_results,
            'lanes': lane_results,
            'warnings': warnings
        }

    def visualize_results(self, result):
        """처리 결과 시각화"""
        # 원본 + 객체 바운딩박스 + 차선 표시 + 위험 구역 색칠
        # (생략: 그리기 코드 동일)
        return original_rgb, result_image


# ============================================
# 2. compare_all_images() - 여러 이미지 처리
# ============================================
def compare_all_images():
    """3장의 이미지를 모두 ADAS로 분석"""
    image_files = ['1.png', '2.png', '3.png']
    available_images = [f'/content/workspace/{f}' for f in image_files if os.path.exists(f'/content/workspace/{f}')]

    if not available_images:
        print("❌ No images found!")
        return

    adas = ComprehensiveADAS()
    all_results = [adas.process_single_image(img) for img in available_images]
    # (이후 matplotlib으로 시각화 및 통계 출력)
    return all_results


# ============================================
# 3. TensorRTADAS - PyTorch vs TensorRT 성능 비교
# ============================================
class TensorRTADAS:
    """TensorRT 최적화 YOLO 모델로 ADAS 수행 + PyTorch 버전 비교"""

    def __init__(self):
        self.load_models()

    def load_models(self):
        """YOLO 모델 로드 & TensorRT 변환 시도"""
        # (모델 로드 + TensorRT export)

    def benchmark_inference(self, image_path, model_type='tensorrt', iterations=20):
        """반복 추론을 통한 FPS, 속도 측정"""
        # (반복 추론 시간 기록 + 평균 FPS 계산)

    def detect_lanes(self, image_path):
        """간단한 차선 탐지"""
        # (Canny + HoughLinesP)

    def compare_performance(self, image_path):
        """동일 이미지에서 PyTorch / TensorRT 성능 비교"""
        # (두 버전의 FPS, 객체 탐지 수, 차선 수 비교)


# ============================================
# 4. create_tensorrt_comparison_charts()
# ============================================
def create_tensorrt_comparison_charts():
    """여러 이미지를 대상으로 TensorRT vs PyTorch 비교 그래프 생성"""
    # (bar chart, scatter plot 등 matplotlib 시각화)


# ============================================
# 5. main() 실행
# ============================================
def main():
    """TensorRT vs PyTorch ADAS 성능 비교 실행"""
    results = create_tensorrt_comparison_charts()
    return results


# ============================================
# 메인 실행
# ============================================
if __name__ == "__main__":
    # 첫 번째 ADAS 실행
    results = compare_all_images()

    # TensorRT 비교 실행
    tensorrt_results = main()


🚗 Initializing Comprehensive ADAS System...
📦 Loading YOLO model...
✅ Model loaded on GPU
🔍 Object Detection: 1.png
🔍 Object Detection: 2.png
🔍 Object Detection: 3.png


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
import torch
from ultralytics import YOLO
import os

class TensorRTADAS:
    """TensorRT 최적화 ADAS 시스템"""

    def __init__(self):
        print("🚀 TensorRT 최적화 YOLO 모델 로딩 중...")
        self.model = None
        self.tensorrt_model = None
        self.load_models()

    def load_models(self):
        """PyTorch 모델과 TensorRT 모델 로드 및 변환 시도"""
        self.model = YOLO('yolov8n.pt')  # 기본 YOLO 모델 로드
        if torch.cuda.is_available():
            self.model.to('cuda')
            print("✅ PyTorch 모델이 GPU에 로드되었습니다")

        try:
            print("🔄 TensorRT 엔진으로 변환 중...")
            self.model.export(format='engine', imgsz=640, half=True, device=0)

            model_name = str(self.model.ckpt_path or 'yolov8n.pt').replace('.pt', '.engine')
            if os.path.exists(model_name):
                self.tensorrt_model = YOLO(model_name)
                print("✅ TensorRT 모델이 성공적으로 로드되었습니다")
            else:
                tensorrt_path = 'yolov8n.engine'
                if os.path.exists(tensorrt_path):
                    self.tensorrt_model = YOLO(tensorrt_path)
                    print("✅ TensorRT 모델이 성공적으로 로드되었습니다")
                else:
                    print("⚠️ TensorRT 엔진 파일을 찾을 수 없어 PyTorch 모델 최적화로 대체합니다...")
                    self.tensorrt_model = self.model
                    self.optimize_pytorch_model()

        except Exception as e:
            print(f"⚠️ TensorRT 변환 실패: {e}")
            print("🔄 PyTorch 최적화 버전 사용합니다...")
            self.tensorrt_model = self.model
            self.optimize_pytorch_model()

    def optimize_pytorch_model(self):
        """TensorRT 미사용 시 PyTorch 최적화 적용"""
        if torch.cuda.is_available():
            torch.backends.cudnn.benchmark = True
            torch.backends.cudnn.deterministic = False
            print("✅ PyTorch 최적화 적용됨 (CUDNN benchmark 활성화)")

        self.tensorrt_model = self.model

    def benchmark_inference(self, image_path, model_type='tensorrt', iterations=20):
        """지정한 모델로 반복 추론 실행 후 성능 측정"""
        model = self.tensorrt_model if model_type == 'tensorrt' else self.model

        print(f"🏃 {model_type.upper()} 모델 벤치마크 ({iterations}회 반복) 시작...")

        # 워밍업 단계
        for _ in range(3):
            _ = model(image_path, conf=0.5, verbose=False)

        times = []
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        for i in range(iterations):
            start_time = time.time()
            results = model(image_path, conf=0.5, verbose=False)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            inference_time = time.time() - start_time
            times.append(inference_time)

            if (i + 1) % 5 == 0:
                current_fps = 1.0 / inference_time
                print(f"    반복 {i+1}: {inference_time:.4f}s ({current_fps:.1f} FPS)")

        avg_time = np.mean(times)
        std_time = np.std(times)
        min_time = np.min(times)
        max_time = np.max(times)
        avg_fps = 1.0 / avg_time
        max_fps = 1.0 / min_time
        min_fps = 1.0 / max_time

        # 탐지 결과 정리
        result = results[0]
        detections = []
        adas_objects = ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 'traffic light', 'stop sign']

        if result.boxes is not None:
            for box in result.boxes:
                class_id = int(box.cls[0])
                class_name = result.names[class_id]
                confidence = float(box.conf[0])
                if class_name in adas_objects:
                    detections.append({
                        'class': class_name,
                        'confidence': confidence
                    })

        return {
            'avg_fps': avg_fps,
            'min_fps': min_fps,
            'max_fps': max_fps,
            'avg_time': avg_time,
            'std_time': std_time,
            'fps_std': std_time * avg_fps * avg_fps,
            'all_times': times,
            'objects': len(detections),
            'detections': detections
        }

    def detect_lanes(self, image_path):
        """간단한 차선 감지"""
        image = cv2.imread(image_path)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        edges = cv2.Canny(blur, 50, 150)

        height, width = gray.shape
        mask = np.zeros_like(edges)
        polygon = np.array([[
            (width // 4, height),
            (width // 2 - 50, height // 2 + 50),
            (width // 2 + 50, height // 2 + 50),
            (3 * width // 4, height)
        ]], np.int32)
        cv2.fillPoly(mask, polygon, 255)
        masked_edges = cv2.bitwise_and(edges, mask)

        lines = cv2.HoughLinesP(masked_edges, 1, np.pi / 180, 50,
                                minLineLength=100, maxLineGap=50)

        return len(lines) if lines is not None else 0

    def compare_performance(self, image_path):
        """같은 이미지에 대해 PyTorch 모델과 TensorRT 모델 성능 비교"""
        print(f"\n🔥 성능 비교: {os.path.basename(image_path)}")
        print("=" * 60)

        pytorch_results = self.benchmark_inference(image_path, 'pytorch', 20)
        tensorrt_results = self.benchmark_inference(image_path, 'tensorrt', 20)
        lanes = self.detect_lanes(image_path)

        fps_improvement = (tensorrt_results['avg_fps'] / pytorch_results['avg_fps'] - 1) * 100
        time_improvement = (pytorch_results['avg_time'] / tensorrt_results['avg_time'] - 1) * 100

        print(f"\n📊 비교 결과:")
        print(f"  PyTorch FPS: {pytorch_results['avg_fps']:.2f} (±{pytorch_results['fps_std']:.2f})")
        print(f"  TensorRT FPS: {tensorrt_results['avg_fps']:.2f} (±{tensorrt_results['fps_std']:.2f})")
        print(f"  🚀 FPS 향상률: {fps_improvement:.1f}%")
        print(f"  ⚡ 처리 시간 단축: {time_improvement:.1f}%")
        print(f"  🎯 탐지 객체 수: PT={pytorch_results['objects']}, TR={tensorrt_results['objects']}")
        print(f"  🛣️ 차선 수: {lanes}")

        return {
            'image_name': os.path.basename(image_path),
            'pytorch': pytorch_results,
            'tensorrt': tensorrt_results,
            'lanes': lanes,
            'fps_improvement': fps_improvement,
            'time_improvement': time_improvement
        }


def create_tensorrt_comparison_charts():
    """TensorRT와 PyTorch 성능 비교 차트 생성"""
    print("🚀 TensorRT와 PyTorch 성능 비교 차트 생성 중...")

    # 이미지 파일명과 경로 - 여기 경로와 파일명은 실제 환경에 맞게 변경하세요
    image_files = ['1.png', '2.png', '3.png']
    base_path = '/content/workspace'
    available_images = [os.path.join(base_path, f) for f in image_files if os.path.exists(os.path.join(base_path, f))]


    if not available_images:
        print("❌ 이미지 파일을 찾을 수 없습니다!")
        return

    adas = TensorRTADAS()
    results = []
    for image_path in available_images:
        result = adas.compare_performance(image_path)
        results.append(result)

    # 차트 그리기 준비
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('TensorRT vs PyTorch ADAS 성능 비교', fontsize=16, fontweight='bold')

    image_names = [r['image_name'] for r in results]
    pytorch_fps = [r['pytorch']['avg_fps'] for r in results]
    tensorrt_fps = [r['tensorrt']['avg_fps'] for r in results]
    fps_improvements = [r['fps_improvement'] for r in results]
    pytorch_objects = [r['pytorch']['objects'] for r in results]
    tensorrt_objects = [r['tensorrt']['objects'] for r in results]
    lanes = [r['lanes'] for r in results]

    # 색상 정의
    pytorch_color = '#FF6B6B'
    tensorrt_color = '#4ECDC4'
    improvement_color = '#45B7D1'

    x = np.arange(len(image_names))
    width = 0.35

    # 1. FPS 성능 비교 막대 그래프
    bars1 = axes[0, 0].bar(x - width / 2, pytorch_fps, width, label='PyTorch', color=pytorch_color, alpha=0.8)
    bars2 = axes[0, 0].bar(x + width / 2, tensorrt_fps, width, label='TensorRT', color=tensorrt_color, alpha=0.8)

    axes[0, 0].set_title('FPS 성능 비교', fontweight='bold')
    axes[0, 0].set_ylabel('초당 프레임 수 (FPS)')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(image_names)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            axes[0, 0].text(bar.get_x() + bar.get_width() / 2., height + 0.5,
                            f'{height:.1f}', ha='center', va='bottom', fontweight='bold')

    # 2. FPS 향상률
    bars = axes[0, 1].bar(image_names, fps_improvements, color=improvement_color, alpha=0.8)
    axes[0, 1].set_title('FPS 향상률 (%)', fontweight='bold')
    axes[0, 1].set_ylabel('향상 비율 (%)')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)

    for bar in bars:
        height = bar.get_height()
        axes[0, 1].text(bar.get_x() + bar.get_width() / 2., height + 1,
                        f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

    # 3. 탐지 객체 수 비교
    bars1 = axes[0, 2].bar(x - width / 2, pytorch_objects, width, label='PyTorch', color=pytorch_color, alpha=0.8)
    bars2 = axes[0, 2].bar(x + width / 2, tensorrt_objects, width, label='TensorRT', color=tensorrt_color, alpha=0.8)

    axes[0, 2].set_title('탐지 객체 수', fontweight='bold')
    axes[0, 2].set_ylabel('탐지된 객체 개수')
    axes[0, 2].set_xticks(x)
    axes[0, 2].set_xticklabels(image_names)
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)

    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            axes[0, 2].text(bar.get_x() + bar.get_width() / 2., height + 0.1,
                            f'{int(height)}', ha='center', va='bottom', fontweight='bold')

    # 4. 처리 시간(ms) 비교
    pytorch_times = [r['pytorch']['avg_time'] * 1000 for r in results]
    tensorrt_times = [r['tensorrt']['avg_time'] * 1000 for r in results]

    bars1 = axes[1, 0].bar(x - width / 2, pytorch_times, width, label='PyTorch', color=pytorch_color, alpha=0.8)
    bars2 = axes[1, 0].bar(x + width / 2, tensorrt_times, width, label='TensorRT', color=tensorrt_color, alpha=0.8)

    axes[1, 0].set_title('처리 시간 비교 (밀리초)', fontweight='bold')
    axes[1, 0].set_ylabel('시간 (ms)')
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels(image_names)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            axes[1, 0].text(bar.get_x() + bar.get_width() / 2., height + 1,
                            f'{height:.1f}', ha='center', va='bottom', fontweight='bold')

    # 5. 요약 텍스트 영역
    axes[1, 1].axis('off')
    avg_pytorch_fps = np.mean(pytorch_fps)
    avg_tensorrt_fps = np.mean(tensorrt_fps)
    avg_improvement = np.mean(fps_improvements)

    summary_text = f"""
텐서RT 최적화 결과
{'='*35}

평균 성능:
• PyTorch FPS: {avg_pytorch_fps:.1f}
• TensorRT FPS: {avg_tensorrt_fps:.1f}
• 평균 향상률: {avg_improvement:.1f}%

최고 성능:
• 최고 FPS: {max(tensorrt_fps):.1f} ({image_names[np.argmax(tensorrt_fps)]})
• 최고 향상률: {max(fps_improvements):.1f}% ({image_names[np.argmax(fps_improvements)]})

탐지 정확도:
• PyTorch 객체 수: {sum(pytorch_objects)}
• TensorRT 객체 수: {sum(tensorrt_objects)}
• 정확도 유지: {'✅' if sum(pytorch_objects) == sum(tensorrt_objects) else '⚠️'}

차선 감지:
• 전체 차선: {sum(lanes)}
• 이미지당 평균: {np.mean(lanes):.1f}
"""

    axes[1, 1].text(0.05, 0.95, summary_text, transform=axes[1, 1].transAxes,
                    fontsize=10, verticalalignment='top', fontfamily='monospace',
                    bbox=dict(boxstyle="round,pad=0.5", facecolor="lightgreen", alpha=0.8))

    # 6. 속도 대 정확도 산점도
    axes[1, 2].scatter(pytorch_fps, pytorch_objects, s=100, color=pytorch_color,
                      alpha=0.8, label='PyTorch', marker='o')
    axes[1, 2].scatter(tensorrt_fps, tensorrt_objects, s=100, color=tensorrt_color,
                      alpha=0.8, label='TensorRT', marker='s')

    for i, name in enumerate(image_names):
        axes[1, 2].annotate(name, (pytorch_fps[i], pytorch_objects[i]),
                           xytext=(5, 5), textcoords='offset points', fontsize=8)
        axes[1, 2].annotate(name, (tensorrt_fps[i], tensorrt_objects[i]),
                           xytext=(5, -10), textcoords='offset points', fontsize=8)

    axes[1, 2].set_title('속도 대 정확도 트레이드오프', fontweight='bold')
    axes[1, 2].set_xlabel('FPS (속도)')
    axes[1, 2].set_ylabel('탐지된 객체 수 (정확도)')
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # 상세 결과 출력
    print(f"\n🏆 TensorRT 최적화 요약")
    print("=" * 60)
    for i, result in enumerate(results):
        print(f"\n{i+1}. {result['image_name']}:")
        print(f"   PyTorch:  {result['pytorch']['avg_fps']:.2f} FPS ({result['pytorch']['avg_time']*1000:.1f}ms)")
        print(f"   TensorRT: {result['tensorrt']['avg_fps']:.2f} FPS ({result['tensorrt']['avg_time']*1000:.1f}ms)")
        print(f"   🚀 향상률: {result['fps_improvement']:.1f}% 증가")
        print(f"   🎯 탐지 객체 수: {result['pytorch']['objects']} → {result['tensorrt']['objects']}")
        print(f"   🛣️ 차선 수: {result['lanes']}")

    print(f"\n🎉 전체 최적화 결과:")
    print(f"   평균 FPS 향상률: {avg_improvement:.1f}%")
    print(f"   최고 단일 향상률: {max(fps_improvements):.1f}%")
    print(f"   정확도 유지 여부: {'예' if sum(pytorch_objects) == sum(tensorrt_objects) else '아니오'}")

    return results

def main():
    """TensorRT vs PyTorch ADAS 성능 비교 메인 함수"""
    print("🚀💨 TensorRT 대 PyTorch ADAS 성능비교 시작")
    print("=" * 60)

    results = create_tensorrt_comparison_charts()

    print("\n✅ TensorRT 최적화 분석 완료!")
    print("🏆 성능 향상 측정 및 시각화 완료!")

    return results

if __name__ == "__main__":
    tensorrt_results = main()


🚀💨 TensorRT 대 PyTorch ADAS 성능비교 시작
🚀 TensorRT와 PyTorch 성능 비교 차트 생성 중...
🚀 TensorRT 최적화 YOLO 모델 로딩 중...
✅ PyTorch 모델이 GPU에 로드되었습니다
🔄 TensorRT 엔진으로 변환 중...
Ultralytics 8.3.176 🚀 Python-3.11.13 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15095MiB)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<1.18.0', 'onnxslim>=0.1.59', 'onnxruntime-gpu'] not found, attempting AutoUpdate...

requirements: AutoUpdate success ✅ 6.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.17.0 opset 19...
ONNX: slimming with onnxslim 0.1.64...
ONNX: export success ✅ 7.6s, saved as 'yolov8n.onnx' (12.2 MB)


자꾸 세션이 중단돼서 다시 해봐야 할 것 같다.